# Fashion-MNIST MLP — Exhaustive Ablation to the Glass Ceiling

A rigorous **6-phase controlled experiment** designed to extract the maximum possible accuracy
from a pure Dense (MLP) network on Fashion-MNIST, one variable at a time.

The central question: **where does the MLP genuinely hit its ceiling?**

Unlike exploratory tuning, an ablation study must change exactly one variable per phase.
Every accuracy delta is then a clean causal statement, not a correlation.

---

## Phase map

| Phase | Variable introduced | New in this phase |
|-------|--------------------|--------------------|
| **P0** — True baseline | Nothing. Fixed arch, deterministic. | Adam · 2×128 · ReLU · no regularization |
| **P1** — Architecture search | Bayesian: layers, neuron width (→512), **GELU/Tanh/ReLU** | No dropout, no BN, Adam fixed |
| **P2** — Dropout | Dropout rate per layer added to search | Everything else from P1 frozen |
| **P3** — Batch Normalization | BN flag per layer added to search | Optimizer still Adam, dropout still searched |
| **P4** — AdamW + weight decay search | Optimizer → AdamW, **weight_decay** added to search | Same space as P3, LR still fixed |
| **P5** — Hyperband + LR search | Search algorithm → Hyperband, **learning_rate** added, warm-started from P4 | Final exhaustion attempt |

**Grand Finale** identifies the true champion across all phases, saves `.keras` + `.weights.h5`,
and presents the per-class error analysis that defines the glass ceiling.

> **Independence guarantee:** every phase section is self-contained — it re-imports, reloads
> data, and redefines all helpers. Jump to any phase and run it cold.


---
## Phase 0 — True Baseline

**Single variable introduced:** none.

A completely fixed, untuned model. Two hidden layers, 128 neurons each, ReLU activation,
Adam with default LR, no dropout, no BatchNorm, no regularization of any kind.

This is the floor everything else is measured against.

> Self-contained — run independently.


In [ ]:
!pip install keras-tuner -q

import math, time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
import keras_tuner as kt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

np.random.seed(123)
tf.random.set_seed(123)

(X_train_full, y_train_full), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()
X_train_full = X_train_full / 255.0
X_test       = X_test  / 255.0
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=123
)
class_names = [
    "T-shirt/top","Trouser","Pullover","Dress","Coat",
    "Sandal","Shirt","Sneaker","Bag","Ankle boot"
]

def lr_schedule_fn(epoch, lr):
    return float(lr) if epoch < 10 else float(lr * math.exp(-0.1))

lr_cb = keras.callbacks.LearningRateScheduler(lr_schedule_fn)

def make_es(patience=5):
    return keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=patience,
        min_delta=0.001, restore_best_weights=True
    )

def safe_get(hps, key, fallback):
    """keras_tuner hp.get() takes only the key. This wraps it safely."""
    try:
        val = hps.get(key)
        return val if val is not None else fallback
    except Exception:
        return fallback

def plot_diagnostics(history, y_pred, title, color="steelblue"):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(title, fontsize=13, fontweight="bold")
    n = len(history.history["loss"])
    x = range(1, n+1)
    axes[0].plot(x, history.history["loss"],     label="Train Loss", color=color,    lw=2)
    axes[0].plot(x, history.history["val_loss"], label="Val Loss",   color="tomato", lw=2)
    axes[0].set_title("Loss vs Epochs"); axes[0].legend(); axes[0].grid(True,ls="--",alpha=0.5)
    axes[1].plot(x, history.history["accuracy"],     label="Train Acc", color=color,    lw=2)
    axes[1].plot(x, history.history["val_accuracy"], label="Val Acc",   color="tomato", lw=2)
    axes[1].set_title("Accuracy vs Epochs"); axes[1].legend()
    axes[1].grid(True,ls="--",alpha=0.5); axes[1].set_ylim(0.5, 1.0)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names,
                ax=axes[2], annot_kws={"size":7})
    axes[2].set_title("Confusion Matrix"); axes[2].tick_params(axis="x", rotation=45)
    plt.tight_layout(); plt.show()
    print(f"Epochs trained: {n}")

WEIGHT_INIT = tf.keras.initializers.HeNormal(seed=123)

print("Setup complete.")
print(f"  Train:{X_train.shape} | Valid:{X_valid.shape} | Test:{X_test.shape}")

# ── Phase 0: fixed 2×128 ReLU, Adam(0.001), no regularization ─────────────────
model_p0 = keras.Sequential([
    keras.Input(shape=(28, 28)),
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation="relu",    kernel_initializer=WEIGHT_INIT),
    keras.layers.Dense(128, activation="relu",    kernel_initializer=WEIGHT_INIT),
    keras.layers.Dense(10,  activation="softmax", kernel_initializer=WEIGHT_INIT),
])
model_p0.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy", metrics=["accuracy"]
)
model_p0.summary()

print("\nTraining Phase 0 (True Baseline)...")
t0 = time.time()
history_p0 = model_p0.fit(
    X_train, y_train, validation_data=(X_valid, y_valid),
    epochs=50, batch_size=256, callbacks=[make_es(), lr_cb], verbose=0
)
print(f"Phase 0 done in {(time.time()-t0)/60:.2f} min.")

loss_p0, acc_p0 = model_p0.evaluate(X_test, y_test, verbose=0)
y_pred_p0 = np.argmax(model_p0.predict(X_test, verbose=0), axis=1)
final_model_p0 = model_p0

print(f"\n{'='*50}")
print(f"  Phase 0  |  Acc: {acc_p0*100:.2f}%  |  Loss: {loss_p0:.4f}")
print(f"  FLOOR — all deltas measured from here.")
print(f"{'='*50}")
print(classification_report(y_test, y_pred_p0, target_names=class_names))

In [ ]:
plot_diagnostics(history_p0, y_pred_p0,
    "Phase 0 — True Baseline (fixed 2×128 ReLU, Adam, no regularization)",
    color="steelblue")

---
## Phase 1 — Architecture Search

**Single variable introduced:** Bayesian hyperparameter search (20 trials) over architecture only.

**New search dimensions vs Phase 0:**
- Number of hidden layers: 1–4
- Neurons per layer: 64 / 128 / 192 / 256 / 320 / 384 / 512  ← wider than before
- Activation per layer: **relu / tanh / gelu**  ← GELU added

**Still fixed (identical to Phase 0):**
- Optimizer: Adam, lr = 0.001
- No dropout, no BatchNorm

**Why GELU now?** GELU is a standard activation in modern dense classifiers
(BERT, GPT, etc.). There is no architectural reason to withhold it until a later phase —
it is purely an architecture choice, not a regularization choice.

**Why neurons up to 512?** The previous 256-neuron ceiling was arbitrary.
Fashion-MNIST has 784 input features and 10 outputs — wider layers may capture
more intermediate structure before regularization narrows it down.

> Self-contained — run independently.


In [ ]:
!pip install keras-tuner -q

import math, time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
import keras_tuner as kt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

np.random.seed(123)
tf.random.set_seed(123)

(X_train_full, y_train_full), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()
X_train_full = X_train_full / 255.0
X_test       = X_test  / 255.0
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=123
)
class_names = [
    "T-shirt/top","Trouser","Pullover","Dress","Coat",
    "Sandal","Shirt","Sneaker","Bag","Ankle boot"
]

def lr_schedule_fn(epoch, lr):
    return float(lr) if epoch < 10 else float(lr * math.exp(-0.1))

lr_cb = keras.callbacks.LearningRateScheduler(lr_schedule_fn)

def make_es(patience=5):
    return keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=patience,
        min_delta=0.001, restore_best_weights=True
    )

def safe_get(hps, key, fallback):
    """keras_tuner hp.get() takes only the key. This wraps it safely."""
    try:
        val = hps.get(key)
        return val if val is not None else fallback
    except Exception:
        return fallback

def plot_diagnostics(history, y_pred, title, color="steelblue"):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(title, fontsize=13, fontweight="bold")
    n = len(history.history["loss"])
    x = range(1, n+1)
    axes[0].plot(x, history.history["loss"],     label="Train Loss", color=color,    lw=2)
    axes[0].plot(x, history.history["val_loss"], label="Val Loss",   color="tomato", lw=2)
    axes[0].set_title("Loss vs Epochs"); axes[0].legend(); axes[0].grid(True,ls="--",alpha=0.5)
    axes[1].plot(x, history.history["accuracy"],     label="Train Acc", color=color,    lw=2)
    axes[1].plot(x, history.history["val_accuracy"], label="Val Acc",   color="tomato", lw=2)
    axes[1].set_title("Accuracy vs Epochs"); axes[1].legend()
    axes[1].grid(True,ls="--",alpha=0.5); axes[1].set_ylim(0.5, 1.0)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names,
                ax=axes[2], annot_kws={"size":7})
    axes[2].set_title("Confusion Matrix"); axes[2].tick_params(axis="x", rotation=45)
    plt.tight_layout(); plt.show()
    print(f"Epochs trained: {n}")

WEIGHT_INIT = tf.keras.initializers.HeNormal(seed=123)

print("Setup complete.")
print(f"  Train:{X_train.shape} | Valid:{X_valid.shape} | Test:{X_test.shape}")

# ── Phase 1: architecture search (layers / width / activation) ─────────────────
def build_p1(hp):
    model = keras.Sequential()
    model.add(keras.Input(shape=(28, 28)))
    model.add(keras.layers.Flatten())
    n = hp.Int("num_layers", 1, 4)
    for i in range(n):
        units = hp.Int(f"units_{i}", min_value=64, max_value=512, step=64)
        act   = hp.Choice(f"act_{i}", ["relu", "tanh", "gelu"])
        model.add(keras.layers.Dense(units, activation=act,
                                     kernel_initializer=WEIGHT_INIT))
    model.add(keras.layers.Dense(10, activation="softmax",
                                  kernel_initializer=WEIGHT_INIT))
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

tuner_p1 = kt.BayesianOptimization(
    build_p1, objective="val_accuracy", max_trials=20,
    directory="glass_ceiling", project_name="p1_architecture",
    overwrite=True, seed=123)

print("Phase 1 Search (Bayesian · 20 trials)")
print("Space: num_layers 1-4 | units 64-512 | act relu/tanh/gelu")
print("Fixed: Adam(0.001), no dropout, no BN")
t0 = time.time()
tuner_p1.search(X_train, y_train, validation_data=(X_valid, y_valid),
                epochs=50, batch_size=256, callbacks=[make_es(), lr_cb], verbose=0)
print(f"Search done in {(time.time()-t0)/60:.2f} min.")

best_p1 = tuner_p1.get_best_hyperparameters(1)[0]
n_p1 = best_p1.get("num_layers")
print(f"\n{'='*50}\n  CHAMPION (PHASE 1)\n{'='*50}")
print(f"  Layers: {n_p1}  |  Optimizer: Adam 0.001 (fixed)")
for i in range(n_p1):
    print(f"  Layer {i+1}: {best_p1.get(f'units_{i}')} × {best_p1.get(f'act_{i}')}")
print("="*50)

final_model_p1 = tuner_p1.hypermodel.build(best_p1)
history_p1 = final_model_p1.fit(
    X_train_full, y_train_full, validation_split=0.2,
    epochs=50, batch_size=256, callbacks=[make_es(), lr_cb], verbose=0)

loss_p1, acc_p1 = final_model_p1.evaluate(X_test, y_test, verbose=0)
y_pred_p1 = np.argmax(final_model_p1.predict(X_test, verbose=0), axis=1)
try:    print(f"  Acc: {acc_p1*100:.2f}%  |  Δ vs P0: {(acc_p1-acc_p0)*100:+.2f}% ← value of search+GELU+width")
except: print(f"  Acc: {acc_p1*100:.2f}%  |  (run P0 for delta)")
print(classification_report(y_test, y_pred_p1, target_names=class_names))

In [ ]:
plot_diagnostics(history_p1, y_pred_p1,
    "Phase 1 — Architecture Search (Bayesian, Adam fixed, no regularization)",
    color="darkorange")

---
## Phase 2 — Dropout Isolation

**Single variable introduced:** Dropout rate per layer added to the search space (0.0 – 0.5).

**Everything else identical to Phase 1:**
- Same Bayesian search (20 trials)
- Same architecture search dimensions (layers 1–4, units 64–512, relu/tanh/gelu)
- Optimizer: Adam(0.001) — unchanged
- No BatchNorm — unchanged

**Question answered:** How much accuracy does dropout contribute, net of architecture search?

> Self-contained — run independently.


In [ ]:
!pip install keras-tuner -q

import math, time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
import keras_tuner as kt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

np.random.seed(123)
tf.random.set_seed(123)

(X_train_full, y_train_full), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()
X_train_full = X_train_full / 255.0
X_test       = X_test  / 255.0
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=123
)
class_names = [
    "T-shirt/top","Trouser","Pullover","Dress","Coat",
    "Sandal","Shirt","Sneaker","Bag","Ankle boot"
]

def lr_schedule_fn(epoch, lr):
    return float(lr) if epoch < 10 else float(lr * math.exp(-0.1))

lr_cb = keras.callbacks.LearningRateScheduler(lr_schedule_fn)

def make_es(patience=5):
    return keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=patience,
        min_delta=0.001, restore_best_weights=True
    )

def safe_get(hps, key, fallback):
    """keras_tuner hp.get() takes only the key. This wraps it safely."""
    try:
        val = hps.get(key)
        return val if val is not None else fallback
    except Exception:
        return fallback

def plot_diagnostics(history, y_pred, title, color="steelblue"):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(title, fontsize=13, fontweight="bold")
    n = len(history.history["loss"])
    x = range(1, n+1)
    axes[0].plot(x, history.history["loss"],     label="Train Loss", color=color,    lw=2)
    axes[0].plot(x, history.history["val_loss"], label="Val Loss",   color="tomato", lw=2)
    axes[0].set_title("Loss vs Epochs"); axes[0].legend(); axes[0].grid(True,ls="--",alpha=0.5)
    axes[1].plot(x, history.history["accuracy"],     label="Train Acc", color=color,    lw=2)
    axes[1].plot(x, history.history["val_accuracy"], label="Val Acc",   color="tomato", lw=2)
    axes[1].set_title("Accuracy vs Epochs"); axes[1].legend()
    axes[1].grid(True,ls="--",alpha=0.5); axes[1].set_ylim(0.5, 1.0)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names,
                ax=axes[2], annot_kws={"size":7})
    axes[2].set_title("Confusion Matrix"); axes[2].tick_params(axis="x", rotation=45)
    plt.tight_layout(); plt.show()
    print(f"Epochs trained: {n}")

WEIGHT_INIT = tf.keras.initializers.HeNormal(seed=123)

print("Setup complete.")
print(f"  Train:{X_train.shape} | Valid:{X_valid.shape} | Test:{X_test.shape}")

# ── Phase 2: + dropout in search space ────────────────────────────────────────
def build_p2(hp):
    model = keras.Sequential()
    model.add(keras.Input(shape=(28, 28)))
    model.add(keras.layers.Flatten())
    n = hp.Int("num_layers", 1, 4)
    for i in range(n):
        units   = hp.Int(f"units_{i}",   min_value=64, max_value=512, step=64)
        act     = hp.Choice(f"act_{i}",  ["relu", "tanh", "gelu"])
        dropout = hp.Float(f"drop_{i}",  min_value=0.0, max_value=0.5, step=0.1)
        model.add(keras.layers.Dense(units, activation=act,
                                     kernel_initializer=WEIGHT_INIT))
        model.add(keras.layers.Dropout(dropout))   # NEW in Phase 2
    model.add(keras.layers.Dense(10, activation="softmax",
                                  kernel_initializer=WEIGHT_INIT))
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

tuner_p2 = kt.BayesianOptimization(
    build_p2, objective="val_accuracy", max_trials=20,
    directory="glass_ceiling", project_name="p2_dropout",
    overwrite=True, seed=123)

print("Phase 2 Search (Bayesian · 20 trials)")
print("Space: same as P1 + dropout 0.0–0.5 per layer  [NEW]")
print("Fixed: Adam(0.001), no BN")
t0 = time.time()
tuner_p2.search(X_train, y_train, validation_data=(X_valid, y_valid),
                epochs=50, batch_size=256, callbacks=[make_es(), lr_cb], verbose=0)
print(f"Search done in {(time.time()-t0)/60:.2f} min.")

best_p2 = tuner_p2.get_best_hyperparameters(1)[0]
n_p2 = best_p2.get("num_layers")
print(f"\n{'='*50}\n  CHAMPION (PHASE 2)\n{'='*50}")
print(f"  Layers: {n_p2}  |  Optimizer: Adam 0.001 (fixed)")
for i in range(n_p2):
    print(f"  Layer {i+1}: {best_p2.get(f'units_{i}')} × {best_p2.get(f'act_{i}')} "
          f"| drop={best_p2.get(f'drop_{i}'):.1f}")
print("="*50)

final_model_p2 = tuner_p2.hypermodel.build(best_p2)
history_p2 = final_model_p2.fit(
    X_train_full, y_train_full, validation_split=0.2,
    epochs=50, batch_size=256, callbacks=[make_es(), lr_cb], verbose=0)

loss_p2, acc_p2 = final_model_p2.evaluate(X_test, y_test, verbose=0)
y_pred_p2 = np.argmax(final_model_p2.predict(X_test, verbose=0), axis=1)
try:
    print(f"  Acc: {acc_p2*100:.2f}%  |  Δ vs P1: {(acc_p2-acc_p1)*100:+.2f}% ← isolated dropout")
    print(f"                        Δ vs P0: {(acc_p2-acc_p0)*100:+.2f}% ← cumulative")
except: print(f"  Acc: {acc_p2*100:.2f}%")
print(classification_report(y_test, y_pred_p2, target_names=class_names))

In [ ]:
plot_diagnostics(history_p2, y_pred_p2,
    "Phase 2 — Dropout Isolation (Bayesian, Adam fixed, dropout searched)",
    color="mediumseagreen")

---
## Phase 3 — Batch Normalization Isolation

**Single variable introduced:** A per-layer Boolean flag `use_bn` added to the search space.

When `use_bn=True` for a layer, a `BatchNormalization()` layer is inserted **after** the
`Dense` and **before** `Dropout`. The tuner can independently choose whether each layer
benefits from BN or not.

**Everything else identical to Phase 2:**
- Same Bayesian search (20 trials)
- Same architecture dimensions and dropout search
- Optimizer: Adam(0.001) — unchanged

**Why BN after Dense but before Dropout?**
The canonical order for classification heads is Dense → BN → Activation → Dropout.
Since activation is fused into `Dense` here, BN goes before Dropout to normalize the
activated outputs before stochastic zeroing.

**Question answered:** How much does BatchNorm contribute, independently of dropout?

> Self-contained — run independently.


In [ ]:
!pip install keras-tuner -q

import math, time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
import keras_tuner as kt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

np.random.seed(123)
tf.random.set_seed(123)

(X_train_full, y_train_full), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()
X_train_full = X_train_full / 255.0
X_test       = X_test  / 255.0
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=123
)
class_names = [
    "T-shirt/top","Trouser","Pullover","Dress","Coat",
    "Sandal","Shirt","Sneaker","Bag","Ankle boot"
]

def lr_schedule_fn(epoch, lr):
    return float(lr) if epoch < 10 else float(lr * math.exp(-0.1))

lr_cb = keras.callbacks.LearningRateScheduler(lr_schedule_fn)

def make_es(patience=5):
    return keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=patience,
        min_delta=0.001, restore_best_weights=True
    )

def safe_get(hps, key, fallback):
    """keras_tuner hp.get() takes only the key. This wraps it safely."""
    try:
        val = hps.get(key)
        return val if val is not None else fallback
    except Exception:
        return fallback

def plot_diagnostics(history, y_pred, title, color="steelblue"):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(title, fontsize=13, fontweight="bold")
    n = len(history.history["loss"])
    x = range(1, n+1)
    axes[0].plot(x, history.history["loss"],     label="Train Loss", color=color,    lw=2)
    axes[0].plot(x, history.history["val_loss"], label="Val Loss",   color="tomato", lw=2)
    axes[0].set_title("Loss vs Epochs"); axes[0].legend(); axes[0].grid(True,ls="--",alpha=0.5)
    axes[1].plot(x, history.history["accuracy"],     label="Train Acc", color=color,    lw=2)
    axes[1].plot(x, history.history["val_accuracy"], label="Val Acc",   color="tomato", lw=2)
    axes[1].set_title("Accuracy vs Epochs"); axes[1].legend()
    axes[1].grid(True,ls="--",alpha=0.5); axes[1].set_ylim(0.5, 1.0)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names,
                ax=axes[2], annot_kws={"size":7})
    axes[2].set_title("Confusion Matrix"); axes[2].tick_params(axis="x", rotation=45)
    plt.tight_layout(); plt.show()
    print(f"Epochs trained: {n}")

WEIGHT_INIT = tf.keras.initializers.HeNormal(seed=123)

print("Setup complete.")
print(f"  Train:{X_train.shape} | Valid:{X_valid.shape} | Test:{X_test.shape}")

# ── Phase 3: + BatchNorm Boolean per layer ────────────────────────────────────
def build_p3(hp):
    model = keras.Sequential()
    model.add(keras.Input(shape=(28, 28)))
    model.add(keras.layers.Flatten())
    n = hp.Int("num_layers", 1, 4)
    for i in range(n):
        units   = hp.Int(f"units_{i}",   min_value=64, max_value=512, step=64)
        act     = hp.Choice(f"act_{i}",  ["relu", "tanh", "gelu"])
        use_bn  = hp.Boolean(f"bn_{i}")                          # NEW in Phase 3
        dropout = hp.Float(f"drop_{i}",  min_value=0.0, max_value=0.5, step=0.1)
        model.add(keras.layers.Dense(units, activation=act,
                                     kernel_initializer=WEIGHT_INIT))
        if use_bn:
            model.add(keras.layers.BatchNormalization())         # NEW in Phase 3
        model.add(keras.layers.Dropout(dropout))
    model.add(keras.layers.Dense(10, activation="softmax",
                                  kernel_initializer=WEIGHT_INIT))
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

tuner_p3 = kt.BayesianOptimization(
    build_p3, objective="val_accuracy", max_trials=20,
    directory="glass_ceiling", project_name="p3_batchnorm",
    overwrite=True, seed=123)

print("Phase 3 Search (Bayesian · 20 trials)")
print("Space: same as P2 + use_bn Boolean per layer  [NEW]")
print("Fixed: Adam(0.001)")
t0 = time.time()
tuner_p3.search(X_train, y_train, validation_data=(X_valid, y_valid),
                epochs=50, batch_size=256, callbacks=[make_es(), lr_cb], verbose=0)
print(f"Search done in {(time.time()-t0)/60:.2f} min.")

best_p3 = tuner_p3.get_best_hyperparameters(1)[0]
n_p3 = best_p3.get("num_layers")
print(f"\n{'='*55}\n  CHAMPION (PHASE 3)\n{'='*55}")
print(f"  Layers: {n_p3}  |  Optimizer: Adam 0.001 (fixed)")
for i in range(n_p3):
    bn = safe_get(best_p3, f"bn_{i}", False)
    print(f"  Layer {i+1}: {safe_get(best_p3,f'units_{i}',128)} × "
          f"{safe_get(best_p3,f'act_{i}','relu')} | "
          f"BN={'yes' if bn else 'no'} | "
          f"drop={safe_get(best_p3,f'drop_{i}',0.0):.1f}")
print("="*55)

final_model_p3 = tuner_p3.hypermodel.build(best_p3)
history_p3 = final_model_p3.fit(
    X_train_full, y_train_full, validation_split=0.2,
    epochs=50, batch_size=256, callbacks=[make_es(), lr_cb], verbose=0)

loss_p3, acc_p3 = final_model_p3.evaluate(X_test, y_test, verbose=0)
y_pred_p3 = np.argmax(final_model_p3.predict(X_test, verbose=0), axis=1)
try:
    print(f"  Acc: {acc_p3*100:.2f}%  |  Δ vs P2: {(acc_p3-acc_p2)*100:+.2f}% ← isolated BN")
    print(f"                        Δ vs P0: {(acc_p3-acc_p0)*100:+.2f}% ← cumulative")
except: print(f"  Acc: {acc_p3*100:.2f}%")
print(classification_report(y_test, y_pred_p3, target_names=class_names))

In [ ]:
plot_diagnostics(history_p3, y_pred_p3,
    "Phase 3 — BatchNorm Isolation (Bayesian, Adam fixed, BN searched)",
    color="mediumpurple")

---
## Phase 4 — AdamW + Weight Decay Search

**Single variable introduced:** Optimizer swapped from Adam to AdamW,
with `weight_decay` added to the search space (log scale: 1e-4 to 1e-1).

**Everything else identical to Phase 3:**
- Same Bayesian search (20 trials)
- Same architecture, dropout, and BN search dimensions
- Learning rate: 0.001 — still fixed

**Why search weight_decay here and not hardcode it?**
AdamW's theoretical advantage over Adam is decoupled weight decay — but that advantage
only materializes at the right decay value. Hardcoding `wd=0.004` (as was done in
the previous study) conflates two things: "does AdamW help?" and "is 0.004 the right
decay?". Searching `weight_decay` ensures we evaluate AdamW at its best,
not at an arbitrary point.

**Question answered:** Does a properly tuned AdamW outperform Adam on this configuration?

> Self-contained — run independently. Stores `p4_defaults` for Phase 5 warm start.


In [ ]:
!pip install keras-tuner -q

import math, time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
import keras_tuner as kt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

np.random.seed(123)
tf.random.set_seed(123)

(X_train_full, y_train_full), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()
X_train_full = X_train_full / 255.0
X_test       = X_test  / 255.0
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=123
)
class_names = [
    "T-shirt/top","Trouser","Pullover","Dress","Coat",
    "Sandal","Shirt","Sneaker","Bag","Ankle boot"
]

def lr_schedule_fn(epoch, lr):
    return float(lr) if epoch < 10 else float(lr * math.exp(-0.1))

lr_cb = keras.callbacks.LearningRateScheduler(lr_schedule_fn)

def make_es(patience=5):
    return keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=patience,
        min_delta=0.001, restore_best_weights=True
    )

def safe_get(hps, key, fallback):
    """keras_tuner hp.get() takes only the key. This wraps it safely."""
    try:
        val = hps.get(key)
        return val if val is not None else fallback
    except Exception:
        return fallback

def plot_diagnostics(history, y_pred, title, color="steelblue"):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(title, fontsize=13, fontweight="bold")
    n = len(history.history["loss"])
    x = range(1, n+1)
    axes[0].plot(x, history.history["loss"],     label="Train Loss", color=color,    lw=2)
    axes[0].plot(x, history.history["val_loss"], label="Val Loss",   color="tomato", lw=2)
    axes[0].set_title("Loss vs Epochs"); axes[0].legend(); axes[0].grid(True,ls="--",alpha=0.5)
    axes[1].plot(x, history.history["accuracy"],     label="Train Acc", color=color,    lw=2)
    axes[1].plot(x, history.history["val_accuracy"], label="Val Acc",   color="tomato", lw=2)
    axes[1].set_title("Accuracy vs Epochs"); axes[1].legend()
    axes[1].grid(True,ls="--",alpha=0.5); axes[1].set_ylim(0.5, 1.0)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names,
                ax=axes[2], annot_kws={"size":7})
    axes[2].set_title("Confusion Matrix"); axes[2].tick_params(axis="x", rotation=45)
    plt.tight_layout(); plt.show()
    print(f"Epochs trained: {n}")

WEIGHT_INIT = tf.keras.initializers.HeNormal(seed=123)

print("Setup complete.")
print(f"  Train:{X_train.shape} | Valid:{X_valid.shape} | Test:{X_test.shape}")

# ── Phase 4: optimizer → AdamW, weight_decay in search space ──────────────────
def build_p4(hp):
    model = keras.Sequential()
    model.add(keras.Input(shape=(28, 28)))
    model.add(keras.layers.Flatten())
    n = hp.Int("num_layers", 1, 4)
    for i in range(n):
        units   = hp.Int(f"units_{i}",   min_value=64, max_value=512, step=64)
        act     = hp.Choice(f"act_{i}",  ["relu", "tanh", "gelu"])
        use_bn  = hp.Boolean(f"bn_{i}")
        dropout = hp.Float(f"drop_{i}",  min_value=0.0, max_value=0.5, step=0.1)
        model.add(keras.layers.Dense(units, activation=act,
                                     kernel_initializer=WEIGHT_INIT))
        if use_bn:
            model.add(keras.layers.BatchNormalization())
        model.add(keras.layers.Dropout(dropout))
    model.add(keras.layers.Dense(10, activation="softmax",
                                  kernel_initializer=WEIGHT_INIT))
    # THE ONLY CHANGE vs Phase 3: AdamW + weight_decay searched
    wd = hp.Float("weight_decay", min_value=1e-4, max_value=1e-1, sampling="log")
    model.compile(
        optimizer=keras.optimizers.AdamW(learning_rate=0.001, weight_decay=wd),
        loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

tuner_p4 = kt.BayesianOptimization(
    build_p4, objective="val_accuracy", max_trials=20,
    directory="glass_ceiling", project_name="p4_adamw",
    overwrite=True, seed=123)

print("Phase 4 Search (Bayesian · 20 trials)")
print("Space: same as P3 + weight_decay 1e-4..1e-1 log  [NEW]")
print("Changed: optimizer Adam → AdamW (weight_decay searched, not hardcoded)")
t0 = time.time()
tuner_p4.search(X_train, y_train, validation_data=(X_valid, y_valid),
                epochs=50, batch_size=256, callbacks=[make_es(), lr_cb], verbose=0)
print(f"Search done in {(time.time()-t0)/60:.2f} min.")

best_p4 = tuner_p4.get_best_hyperparameters(1)[0]
n_p4 = best_p4.get("num_layers")
wd_p4 = safe_get(best_p4, "weight_decay", 0.004)
print(f"\n{'='*55}\n  CHAMPION (PHASE 4)\n{'='*55}")
print(f"  Layers: {n_p4}  |  AdamW  wd={wd_p4:.5f}  [CHANGED]")
for i in range(n_p4):
    bn = safe_get(best_p4, f"bn_{i}", False)
    print(f"  Layer {i+1}: {safe_get(best_p4,f'units_{i}',128)} × "
          f"{safe_get(best_p4,f'act_{i}','relu')} | "
          f"BN={'yes' if bn else 'no'} | "
          f"drop={safe_get(best_p4,f'drop_{i}',0.0):.1f}")
print("="*55)

# ── Store config for Phase 5 warm-start ───────────────────────────────────────
p4_defaults = {"num_layers": n_p4, "weight_decay": wd_p4}
for i in range(n_p4):
    p4_defaults[f"units_{i}"]  = safe_get(best_p4, f"units_{i}", 256)
    p4_defaults[f"act_{i}"]    = safe_get(best_p4, f"act_{i}",   "relu")
    p4_defaults[f"bn_{i}"]     = bool(safe_get(best_p4, f"bn_{i}", False))
    p4_defaults[f"drop_{i}"]   = safe_get(best_p4, f"drop_{i}",  0.2)
for i in range(n_p4, 4):
    p4_defaults[f"units_{i}"] = 128; p4_defaults[f"act_{i}"] = "relu"
    p4_defaults[f"bn_{i}"]    = False; p4_defaults[f"drop_{i}"] = 0.2
print("\np4_defaults stored → seeds Phase 5 warm start.")
print(p4_defaults)

final_model_p4 = tuner_p4.hypermodel.build(best_p4)
history_p4 = final_model_p4.fit(
    X_train_full, y_train_full, validation_split=0.2,
    epochs=50, batch_size=256, callbacks=[make_es(), lr_cb], verbose=0)

loss_p4, acc_p4 = final_model_p4.evaluate(X_test, y_test, verbose=0)
y_pred_p4 = np.argmax(final_model_p4.predict(X_test, verbose=0), axis=1)
try:
    print(f"  Acc: {acc_p4*100:.2f}%  |  Δ vs P3: {(acc_p4-acc_p3)*100:+.2f}% ← isolated AdamW+wd")
    print(f"                        Δ vs P0: {(acc_p4-acc_p0)*100:+.2f}% ← cumulative")
except: print(f"  Acc: {acc_p4*100:.2f}%")
print(classification_report(y_test, y_pred_p4, target_names=class_names))

In [ ]:
plot_diagnostics(history_p4, y_pred_p4,
    "Phase 4 — AdamW + Weight Decay Search (Bayesian, LR still fixed)",
    color="chocolate")

---
## Phase 5 — Hyperband + Learning Rate Search (Final Exhaustion)

**Single variable introduced:** The search algorithm upgrades to Hyperband (Successive Halving),
and `learning_rate` is added to the search space (log scale: 1e-4 to 1e-2).

**Everything else identical to Phase 4:**
- Same architecture, dropout, BN, and weight_decay search dimensions
- Optimizer: AdamW — unchanged
- Warm-started from Phase 4's champion configuration

**Why LR here?**
LR is the last fixed hyperparameter that was never searched. By isolating it to Phase 5,
we keep the causal chain clean: every prior phase's delta is attributable to its single variable.

**Why Hyperband now?**
Hyperband's successive halving is most valuable when the search space is large.
After five phases of pruning, the space is well-understood — the warm start ensures
Hyperband's initial bracket is anchored near the best-known region.

**Question answered:** Is there any residual accuracy left after all regularization,
normalization, optimizer, and architecture choices have been made?

> Self-contained — run independently.
> ⚠️  For proper warm start, run Phase 4 first so `p4_defaults` is in session.
> If running cold, the fallback config at the top of this cell will be used.


In [ ]:
!pip install keras-tuner -q

import math, time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
import keras_tuner as kt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

np.random.seed(123)
tf.random.set_seed(123)

(X_train_full, y_train_full), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()
X_train_full = X_train_full / 255.0
X_test       = X_test  / 255.0
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=123
)
class_names = [
    "T-shirt/top","Trouser","Pullover","Dress","Coat",
    "Sandal","Shirt","Sneaker","Bag","Ankle boot"
]

def lr_schedule_fn(epoch, lr):
    return float(lr) if epoch < 10 else float(lr * math.exp(-0.1))

lr_cb = keras.callbacks.LearningRateScheduler(lr_schedule_fn)

def make_es(patience=5):
    return keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=patience,
        min_delta=0.001, restore_best_weights=True
    )

def safe_get(hps, key, fallback):
    """keras_tuner hp.get() takes only the key. This wraps it safely."""
    try:
        val = hps.get(key)
        return val if val is not None else fallback
    except Exception:
        return fallback

def plot_diagnostics(history, y_pred, title, color="steelblue"):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(title, fontsize=13, fontweight="bold")
    n = len(history.history["loss"])
    x = range(1, n+1)
    axes[0].plot(x, history.history["loss"],     label="Train Loss", color=color,    lw=2)
    axes[0].plot(x, history.history["val_loss"], label="Val Loss",   color="tomato", lw=2)
    axes[0].set_title("Loss vs Epochs"); axes[0].legend(); axes[0].grid(True,ls="--",alpha=0.5)
    axes[1].plot(x, history.history["accuracy"],     label="Train Acc", color=color,    lw=2)
    axes[1].plot(x, history.history["val_accuracy"], label="Val Acc",   color="tomato", lw=2)
    axes[1].set_title("Accuracy vs Epochs"); axes[1].legend()
    axes[1].grid(True,ls="--",alpha=0.5); axes[1].set_ylim(0.5, 1.0)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names,
                ax=axes[2], annot_kws={"size":7})
    axes[2].set_title("Confusion Matrix"); axes[2].tick_params(axis="x", rotation=45)
    plt.tight_layout(); plt.show()
    print(f"Epochs trained: {n}")

WEIGHT_INIT = tf.keras.initializers.HeNormal(seed=123)

print("Setup complete.")
print(f"  Train:{X_train.shape} | Valid:{X_valid.shape} | Test:{X_test.shape}")

# ── Phase 5: Hyperband + LR search, warm-started from Phase 4 ─────────────────
# Fallback if running this phase in isolation (replace with your P4 champion):
if "p4_defaults" not in dir():
    print("WARNING: p4_defaults not found. Using generic fallback.")
    p4_defaults = {
        "num_layers": 2, "weight_decay": 0.004,
        "units_0": 256, "act_0": "gelu", "bn_0": True,  "drop_0": 0.2,
        "units_1": 256, "act_1": "relu", "bn_1": False, "drop_1": 0.1,
        "units_2": 128, "act_2": "relu", "bn_2": False, "drop_2": 0.2,
        "units_3": 128, "act_3": "relu", "bn_3": False, "drop_3": 0.2,
    }
print("Warm-start config:", p4_defaults)

def build_p5(hp):
    model = keras.Sequential()
    model.add(keras.Input(shape=(28, 28)))
    model.add(keras.layers.Flatten())
    n = hp.Int("num_layers", 1, 4,
                default=p4_defaults["num_layers"])
    for i in range(n):
        units   = hp.Int(f"units_{i}",  64, 512, step=64,
                          default=int(p4_defaults.get(f"units_{i}", 256)))
        act     = hp.Choice(f"act_{i}", ["relu","tanh","gelu"],
                             default=str(p4_defaults.get(f"act_{i}", "relu")))
        use_bn  = hp.Boolean(f"bn_{i}",
                              default=bool(p4_defaults.get(f"bn_{i}", False)))
        dropout = hp.Float(f"drop_{i}", 0.0, 0.5, step=0.1,
                            default=float(p4_defaults.get(f"drop_{i}", 0.2)))
        model.add(keras.layers.Dense(units, activation=act,
                                     kernel_initializer=WEIGHT_INIT))
        if use_bn:
            model.add(keras.layers.BatchNormalization())
        model.add(keras.layers.Dropout(dropout))
    model.add(keras.layers.Dense(10, activation="softmax",
                                  kernel_initializer=WEIGHT_INIT))
    # Learning rate now searched — THE ONLY NEW DIMENSION vs Phase 4
    lr  = hp.Float("learning_rate", 1e-4, 1e-2, sampling="log",
                    default=0.001)
    wd  = hp.Float("weight_decay",  1e-4, 1e-1, sampling="log",
                    default=float(p4_defaults.get("weight_decay", 0.004)))
    model.compile(
        optimizer=keras.optimizers.AdamW(learning_rate=lr, weight_decay=wd),
        loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

tuner_p5 = kt.Hyperband(
    build_p5, objective="val_accuracy",
    max_epochs=50, factor=3,
    directory="glass_ceiling", project_name="p5_hyperband_lr",
    overwrite=True, seed=123)

print("\nPhase 5 Search (Hyperband · max_epochs=50 · factor=3)")
print("Space: same as P4 + learning_rate 1e-4..1e-2 log  [NEW]")
print("Algorithm: Hyperband (warm-started from P4 champion)")
t0 = time.time()
tuner_p5.search(X_train, y_train, validation_data=(X_valid, y_valid),
                epochs=50, batch_size=256, callbacks=[make_es(patience=4), lr_cb], verbose=0)
print(f"Search done in {(time.time()-t0)/60:.2f} min.")

best_p5 = tuner_p5.get_best_hyperparameters(1)[0]
n_p5 = best_p5.get("num_layers")
lr_p5 = safe_get(best_p5, "learning_rate", 0.001)
wd_p5 = safe_get(best_p5, "weight_decay",  0.004)
print(f"\n{'='*55}\n  CHAMPION (PHASE 5)\n{'='*55}")
print(f"  Layers: {n_p5}  |  AdamW  lr={lr_p5:.5f}  wd={wd_p5:.5f}  [CHANGED]")
for i in range(n_p5):
    bn = safe_get(best_p5, f"bn_{i}", False)
    print(f"  Layer {i+1}: {safe_get(best_p5,f'units_{i}',128)} × "
          f"{safe_get(best_p5,f'act_{i}','relu')} | "
          f"BN={'yes' if bn else 'no'} | "
          f"drop={safe_get(best_p5,f'drop_{i}',0.0):.1f}")
print("="*55)

final_model_p5 = tuner_p5.hypermodel.build(best_p5)
history_p5 = final_model_p5.fit(
    X_train_full, y_train_full, validation_split=0.2,
    epochs=50, batch_size=256,
    callbacks=[make_es(patience=5), lr_cb], verbose=0)

loss_p5, acc_p5 = final_model_p5.evaluate(X_test, y_test, verbose=0)
y_pred_p5 = np.argmax(final_model_p5.predict(X_test, verbose=0), axis=1)
try:
    print(f"  Acc: {acc_p5*100:.2f}%  |  Δ vs P4: {(acc_p5-acc_p4)*100:+.2f}% ← isolated LR+Hyperband")
    print(f"                        Δ vs P0: {(acc_p5-acc_p0)*100:+.2f}% ← total cumulative gain")
except: print(f"  Acc: {acc_p5*100:.2f}%")
print(classification_report(y_test, y_pred_p5, target_names=class_names))

In [ ]:
plot_diagnostics(history_p5, y_pred_p5,
    "Phase 5 — Hyperband + LR Search (final exhaustion, warm-started from P4)",
    color="crimson")

---
## Grand Finale — Glass Ceiling Analysis & Champion Model

> Requires all six phases to have been run in this session.
> The champion is determined programmatically — never hardcoded.


In [ ]:
# ── Grand Finale: full comparison, glass ceiling analysis, save champion ───────

PHASES = {
    "P0 — Baseline":           {"acc": acc_p0, "loss": loss_p0, "history": history_p0,
                                 "color": "steelblue",     "pred": y_pred_p0},
    "P1 — Architecture":       {"acc": acc_p1, "loss": loss_p1, "history": history_p1,
                                 "color": "darkorange",    "pred": y_pred_p1},
    "P2 — Dropout":            {"acc": acc_p2, "loss": loss_p2, "history": history_p2,
                                 "color": "mediumseagreen","pred": y_pred_p2},
    "P3 — BatchNorm":          {"acc": acc_p3, "loss": loss_p3, "history": history_p3,
                                 "color": "mediumpurple",  "pred": y_pred_p3},
    "P4 — AdamW+WD":           {"acc": acc_p4, "loss": loss_p4, "history": history_p4,
                                 "color": "chocolate",     "pred": y_pred_p4},
    "P5 — Hyperband+LR":       {"acc": acc_p5, "loss": loss_p5, "history": history_p5,
                                 "color": "crimson",       "pred": y_pred_p5},
}

# ── 1. Summary table ──────────────────────────────────────────────────────────
base = PHASES["P0 — Baseline"]["acc"]
best_name = max(PHASES, key=lambda k: PHASES[k]["acc"])
print("\n" + "="*72)
print(f"  {'Phase':<30} {'Test Acc':>10} {'Test Loss':>12} {'vs P0':>8}")
print("="*72)
for name, v in PHASES.items():
    delta = (v["acc"] - base) * 100
    mark  = "  ★ BEST" if name == best_name else ""
    print(f"  {name:<30} {v['acc']*100:>9.2f}%  {v['loss']:>10.4f}  {delta:>+7.2f}%{mark}")
print("="*72)

# ── 2. Overlay curves ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("All Phases — Validation Curves Overlay", fontsize=14, fontweight="bold")
for name, v in PHASES.items():
    short = name.split("—")[0].strip()
    axes[0].plot(v["history"].history["val_loss"],
                 label=short, color=v["color"], lw=2)
    axes[1].plot(v["history"].history["val_accuracy"],
                 label=short, color=v["color"], lw=2)
axes[0].set_title("Validation Loss (lower is better)")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].legend(fontsize=9); axes[0].grid(True, ls="--", alpha=0.5)
axes[1].set_title("Validation Accuracy (higher is better)")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
axes[1].legend(fontsize=9); axes[1].grid(True, ls="--", alpha=0.5)
axes[1].set_ylim(0.7, 1.0)
plt.tight_layout(); plt.show()

# ── 3. Bar chart ──────────────────────────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(13, 5))
names  = list(PHASES.keys())
accs   = [v["acc"]*100 for v in PHASES.values()]
colors = [v["color"]   for v in PHASES.values()]
bars   = ax2.bar(names, accs, color=colors, alpha=0.85,
                  edgecolor="white", linewidth=0.8)
ax2.set_ylim(min(accs) - 2, max(accs) + 2)
ax2.axhline(y=base*100, color="steelblue", ls="--", alpha=0.4, label="P0 baseline")
ax2.axhline(y=91.5, color="gray", ls=":", alpha=0.6, label="~MLP ceiling (literature ~91-91.5%)")
for bar, acc in zip(bars, accs):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.08,
             f"{acc:.2f}%", ha="center", va="bottom", fontsize=9, fontweight="bold")
ax2.set_title("Test Accuracy by Phase — Each Bar Is One Isolated Variable",
              fontsize=12, fontweight="bold")
ax2.set_ylabel("Test Accuracy (%)"); ax2.legend()
plt.xticks(rotation=15, ha="right"); plt.tight_layout(); plt.show()

# ── 4. Glass ceiling: per-class error analysis ────────────────────────────────
print("\n=== GLASS CEILING ANALYSIS: per-class F1 across all phases ===")
print(f"{'Class':<18}", end="")
for name in PHASES:
    print(f"  {name.split()[0]+name.split()[1]:>10}", end="")
print()
print("-"*90)

from sklearn.metrics import f1_score
for cls_idx, cls_name in enumerate(class_names):
    print(f"{cls_name:<18}", end="")
    for v in PHASES.values():
        f1 = f1_score(y_test, v["pred"], labels=[cls_idx], average="macro")
        print(f"  {f1*100:>10.1f}", end="")
    print()
print()
print("Classes with F1 < 80% across ALL phases = structural MLP limitations.")
print("These are the garments that look identical without spatial features.")

# ── 5. Determine champion and save ────────────────────────────────────────────
model_lookup = {
    "P0 — Baseline":     globals().get("final_model_p0"),
    "P1 — Architecture": globals().get("final_model_p1"),
    "P2 — Dropout":      globals().get("final_model_p2"),
    "P3 — BatchNorm":    globals().get("final_model_p3"),
    "P4 — AdamW+WD":     globals().get("final_model_p4"),
    "P5 — Hyperband+LR": globals().get("final_model_p5"),
}
available = {k: v for k, v in model_lookup.items() if v is not None}
if not available:
    raise RuntimeError("No final_model_pX found. Run at least one phase first.")

champion_name  = max(available, key=lambda k: PHASES[k]["acc"])
champion_model = available[champion_name]
champion_acc   = PHASES[champion_name]["acc"]

print(f"\n★  CHAMPION : {champion_name}")
print(f"   Accuracy  : {champion_acc*100:.2f}%")
print(f"   Total gain: +{(champion_acc - base)*100:.2f}% over untuned baseline")
print(f"   Sessions  : {list(available.keys())}")

champion_model.save("fashion_mnist_mlp_champion.keras")
champion_model.save_weights("fashion_mnist_mlp_champion.weights.h5")
print("\n[SAVED] fashion_mnist_mlp_champion.keras")
print("[SAVED] fashion_mnist_mlp_champion.weights.h5")

print("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Transfer learning to a different dataset
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  Option A — replace output layer, fine-tune:
    base = tf.keras.models.load_model("fashion_mnist_mlp_champion.keras")
    x      = base.layers[-2].output
    out    = keras.layers.Dense(N_CLASSES, activation="softmax")(x)
    model  = keras.Model(base.input, out)
    for layer in model.layers[:-2]:
        layer.trainable = False          # optional: freeze body
    model.compile(optimizer=keras.optimizers.AdamW(learning_rate=1e-4),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])

  Option B — weights only (same architecture required):
    model.load_weights("fashion_mnist_mlp_champion.weights.h5")
""")

---
## Conclusion — The MLP Glass Ceiling

This study now answers Q2 definitively: **no, there is nothing left to tune within the
pure Dense architecture constraint**.

Every meaningful axis has been searched in isolation:

| What was searched | Phase that isolated it |
|-------------------|----------------------|
| Architecture (layers, width 64-512, activation relu/tanh/gelu) | P1 |
| Dropout rate per layer (0.0–0.5) | P2 |
| Batch Normalization per layer (on/off) | P3 |
| Optimizer (AdamW) + weight decay (log search) | P4 |
| Learning rate (log search) + Hyperband algorithm | P5 |

The remaining errors — consistently concentrated in **Shirt, Coat, and Pullover** —
are not tuning failures. They are structural: these garments differ primarily in
local texture, drape, and spatial cut. A flattened 784-pixel input treats every pixel
as an independent, position-agnostic feature. There is no representation of
"the collar is here, the hem is there" — only global intensity statistics.

CNNs reach **95–96%** on this dataset with simple architectures because they build
exactly those spatial features through their inductive bias: translation equivariance,
local receptive fields, hierarchical composition. The ~4–5% gap between the MLP ceiling
and a baseline CNN is **not a tuning gap** — it is an **architectural gap**, and
the per-class F1 table above shows exactly which categories it lives in.

This is the glass ceiling.
